In [61]:
import sys
from pathlib import Path

project_root = Path.cwd().resolve().parent

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

In [62]:
import torch
import torch.nn as nn
import torch.functional as F
import numpy
import math

from tokenizer import CharacterTokenizer

### Tokenization and Embeddings

Tokenization will be done based on a character-level tokenizer.

In [63]:
text = "Hello! I am learning how to build a transformer from scratch using PyTorch."

tokenizer = CharacterTokenizer(text)

tokens = tokenizer.encode(text)

print(tokenizer.decode(tokens) == text)


True


The dimensions of the embeddings will be
$$
E_{sum}=E_{token}+E_{position}
$$

The shape of these tensors are as follows
- $E_{sum}$ : $x_1 + x_2 \in \mathbb{R^{T \times {d_{embed}}}}$
- $E_{token}$ and $E_{position}$ : $x \in \mathbb{R^{T \times {d_{embed}}}}$

where $T$ is the size of the sequence of tokens


In [50]:
tokens = torch.tensor(tokenizer.encode(text), dtype=torch.long)
d_embed = 64

embedding_matrix = torch.randn(tokenizer.vocab_size,d_embed)
pos_embedding_matrix = torch.randn(128, d_embed)
positions = torch.arange(len(tokens))

embeddings = embedding_matrix[tokens]
embedded_tokens = pos_embedding_matrix[positions] + embeddings

print(pos_embedding_matrix.shape)
print(embedded_tokens.shape)
embedded_tokens

torch.Size([128, 64])
torch.Size([75, 64])


tensor([[-1.8283,  0.0280,  0.3867,  ...,  0.0080, -1.2732, -0.7552],
        [-0.3399, -0.6439,  0.1330,  ..., -3.4118, -3.0209, -2.1692],
        [ 1.7092,  2.4804, -0.3583,  ..., -0.1211,  0.4416, -0.1870],
        ...,
        [-1.3956, -0.5667,  1.1451,  ...,  0.7165,  0.6075, -0.1522],
        [-1.5331, -2.0800, -1.9508,  ..., -0.7740, -2.6663,  3.2516],
        [ 0.0357,  1.2377,  1.0899,  ..., -1.0326,  0.4854, -0.9305]])

### Self-Attention Mechanism

We compute the self-attention via computing 3 representations: $W_q, W_k$ and $W_v$



Then, we calculate the scaled dot product and apply a causal mask. The formula is:
$$
S=\frac{QK^T}{\sqrt{d_k}}
$$

After obtaining this result, we apply the softmax multiply by the value vector
$$
Attention(Q, K, V)=Softmax(\frac{QK^T}{\sqrt{d_k}})V
$$

In [51]:
w_q, w_k, w_v = (torch.randn(64, 64), 
                 torch.randn(64, 64), 
                 torch.randn(64, 64))

query_vector = torch.matmul(embedded_tokens, w_q)
key_vector = torch.matmul(embedded_tokens, w_k)
value_vector = torch.matmul(embedded_tokens, w_v)

scaled_dot_product = torch.matmul(query_vector, key_vector.T) / math.sqrt(d_embed)

triu_rows, triu_cols = torch.triu_indices(row=scaled_dot_product.size(0), 
                                  col=scaled_dot_product.size(1),
                                  offset=1)

scaled_dot_product[triu_rows, triu_cols] = float("-inf")

scaled_dot_product.shape
soft_max = nn.Softmax(dim=1)

out = soft_max(scaled_dot_product)
print(out.sum(dim=1) == torch.ones(out.shape[0], out.shape[1]))

A = torch.matmul(out, value_vector)
print(f"Output Shape: {A.shape} \n{A}")



tensor([[ True,  True,  True,  ...,  True, False,  True],
        [ True,  True,  True,  ...,  True, False,  True],
        [ True,  True,  True,  ...,  True, False,  True],
        ...,
        [ True,  True,  True,  ...,  True, False,  True],
        [ True,  True,  True,  ...,  True, False,  True],
        [ True,  True,  True,  ...,  True, False,  True]])
Output Shape: torch.Size([75, 64]) 
tensor([[  6.2718,  22.5267,   2.4609,  ..., -10.8129,   2.3752,  -1.5687],
        [  6.3741, -10.4531, -22.2229,  ..., -10.2377,  -6.3076,  30.3442],
        [  6.2718,  22.5267,   2.4609,  ..., -10.8129,   2.3752,  -1.5687],
        ...,
        [ -8.6354,  10.8891,   4.1936,  ...,  12.6661,  -2.6389,   4.5424],
        [ 15.5883, -18.5458,  16.0343,  ...,   9.9891,  -9.7664,   6.2027],
        [  4.1148,   0.2401,  -9.5836,  ..., -13.5647,  -8.7939,  -8.8784]])


### Multi-Head Attention

In [52]:
class MultiHeadAttention(nn.Module):
    def __init__(self, count_heads, d_embed):
        super().__init__()
        self.heads = count_heads
        self.d_embed = d_embed
        self.d_head = d_embed // count_heads

        self.w_q = nn.ParameterList(
            [nn.Parameter(torch.randn(d_embed, self.d_head)) for _ in range(self.heads)])

        self.w_k = nn.ParameterList(
                [nn.Parameter(torch.randn(d_embed, self.d_head)) for _ in range(self.heads)])

        self.w_v = nn.ParameterList(
                [nn.Parameter(torch.randn(d_embed, self.d_head)) for _ in range(self.heads)])
        
        self.wO = nn.Parameter(torch.randn(d_embed, d_embed))


    def forward(self, X):
        query_vector: list = [torch.matmul(X, w_q) for w_q in self.w_q]
        key_vector: list = [torch.matmul(X, w_k) for w_k in self.w_k]
        value_vector: list = [torch.matmul(X, w_v) for w_v in self.w_v]
        T: int = X.size(0)

        head_outputs = [self.__attention(query_vector[i], key_vector[i], value_vector[i], T) for i in range(self.heads)]
        attention_concat = torch.cat(head_outputs, dim=1)

        return torch.matmul(attention_concat, self.wO)


    def __attention(self, q_i, k_i, v_i, T):
        scaled_dot_product = torch.matmul(q_i, k_i.T) / math.sqrt(self.d_head)

        mask = torch.triu(
            torch.ones(T, T, dtype=torch.bool),
            diagonal=1
        )
        masked_scores = scaled_dot_product.masked_fill(mask, float("-inf"))

        soft_max = nn.Softmax(dim=1)
        out = soft_max(masked_scores)

        attention_matrix = torch.matmul(out, v_i)

        return attention_matrix



In [54]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model = MultiHeadAttention(4, 64).to(device)
print(model)

out = model(embedded_tokens)
print(f"Output Shape: {out.shape}\n{out}")

MultiHeadAttention(
  (w_q): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
  (w_k): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
  (w_v): ParameterList(
      (0): Parameter containing: [torch.float32 of size 64x16]
      (1): Parameter containing: [torch.float32 of size 64x16]
      (2): Parameter containing: [torch.float32 of size 64x16]
      (3): Parameter containing: [torch.float32 of size 64x16]
  )
)
Output Shape: torch.Size([75, 64])
tensor([[  28.9681, -111.8585,  -94.7072,  ...,  -46.5071,   66.2068,
           56.9234],
        [   